# Capa Silver - Catálogo de Clientes

## Librerias

In [1]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Variables

In [2]:
%run ../00-Modulos/n_modulo_variables.ipynb

Servicios:
 Warehouse  = s3a://iceberg/warehouse
 EndPoint   = http://localhost:9000
 Uri        = thrift://localhost:9083
Base de Datos Landing =  iceberg.landing
Base de Datos Bronze  =  iceberg.bronze
Base de Datos Silver  =  iceberg.silver
Base de Datos Gold    =  iceberg.gold


## Spark Session

In [3]:
spark = spark_session_hive("Lakehouse-Iceberg")

## Extracción

In [4]:
df_customer = spark.read.table(f'{vSchemaBron}.customer')

## Transformaciones

In [5]:
# Filtro de clientes validos
df_filter = df_customer.where((f.col("CUSTOMER_NO").isNotNull()) & (f.col("CUSTOMER_NO") != 99999))

# Filtro de clientes unicos
df_filter = df_filter.withColumn("ROW_NUM", f.row_number().over(Window.partitionBy(f.col("CUSTOMER_NO")).orderBy(f.col("CUST_DATE").desc())))
df_customer_unique = df_filter.where(f.col("ROW_NUM") == 1)

# Generación de columnas calculadas
df_customer_unique = df_customer_unique \
    .withColumn("CUST_NAME", combinar_nombre_apellido_udf(f.col("FIRSTNAME"), f.col("LASTNAME"))) \
    .withColumn("CUST_PHONE_NMBR", formato_telefono_udf(f.col("PHONE_NUMBER"))) \
    .withColumn("CUST_GENDER", genero_completo_udf(f.col("GENDER"))) \
    .withColumn("CUST_AGE_GROUP", grupo_edad_udf(f.col("AGE")))  

# Seleccion de columnas finales
df_source = df_customer_unique.select(
    f.col("CUSTOMER_NO").alias("CUST_ID"),
    f.col("CUST_NAME"),
    f.col("ADDRESS").alias("CUST_ADDRESS"),
    f.col("CITY").alias("CUST_CITY"),
    f.col("STATE").alias("CUST_STATE"),
    f.col("ZIP").alias("CUST_ZIP_CODE"),
    f.col("COUNTRY").alias("CUST_COUNTRY"),
    f.col("CUST_PHONE_NMBR"),
    f.col("CUST_GENDER"),
    f.col("CUST_AGE_GROUP"),
    f.col("INCOME").alias("CUST_INCOME"),
    f.col("EMAIL").alias("CUST_E_MAIL"),
    f.col("AGE").alias("CUST_AGE")
)

## Carga

In [6]:
# Obtenemos variables para merge
target_table = f'{vSchemaSilv}.customers'
join_columns = ["CUST_ID"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)